# Exact SU(4) fourth-order completion

This is a complete, sequential Colab calculation of the exceptional-rank fourth-order one-flux $T_1^{+-}$ band for Hamiltonian $SU(4)$ lattice gauge theory.

It performs, without manual edits:

1. primitive Stage-0 input verification;
2. a prefix-aware modulo-four scan of all 182,440 supports and 895,524 support/output pairs;
3. exact pairing-plus-$\epsilon_{i_1i_2i_3i_4}$ local invariant quotients;
4. all 1,806 exceptional fusion-path contractions;
5. an independent 35,130-path balanced fixed-rank contraction at $N=4$;
6. 30 exact cross-engine topology regressions;
7. construction of the corrected 189-record real-space kernel;
8. the exact 25-point Laurent/SOS identity and global band-edge theorem.

Each heavy phase runs in a fresh Python subprocess and exchanges only hashed JSON artifacts. This avoids state or memory contamination between the explicit epsilon-tensor engine and the balanced walled-Brauer engine.

**Required input:** either `SU6_DETERMINANT_ARCHIVAL_V2_BUNDLE.zip` or `GLUEBALL_FLAT_BAND_SOURCE_RELEASE_V0_9_1.zip`. The first cell locates it automatically or opens a Colab upload dialog.

In [ ]:
# 1. Environment and source-bundle discovery
from __future__ import annotations
import os, sys, json, zipfile, shutil, hashlib, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
BASE = Path('/content') if Path('/content').exists() else Path('/mnt/data')
WORK = BASE / 'SU4_EXACT_FOURTH_ORDER'
INPUT = WORK / 'inputs'
OUTPUT = WORK / 'outputs'
STAGES = WORK / 'stages'
LOGS = WORK / 'logs'

for path in (WORK, INPUT, OUTPUT, STAGES, LOGS):
    if path != WORK and path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def discover():
    roots = [BASE, Path('/mnt/data'), Path.cwd()]
    direct, releases = [], []
    for root in roots:
        if not root.exists():
            continue
        direct.extend(root.glob('SU6_DETERMINANT_ARCHIVAL_V2_BUNDLE*.zip'))
        releases.extend(root.glob('GLUEBALL_FLAT_BAND_SOURCE_RELEASE_V0_9_1*.zip'))
    unique = lambda xs: sorted(set(p.resolve() for p in xs), key=str)
    return unique(direct) + unique(releases)

candidates = discover()
if not candidates and IN_COLAB:
    from google.colab import files
    print('Upload SU6_DETERMINANT_ARCHIVAL_V2_BUNDLE.zip or GLUEBALL_FLAT_BAND_SOURCE_RELEASE_V0_9_1.zip')
    uploaded = files.upload()
    candidates = [BASE / name for name in uploaded]
if not candidates:
    raise FileNotFoundError('Required archival source bundle was not found.')

SOURCE_ZIP = candidates[0]
print('SOURCE', SOURCE_ZIP)
print('SHA256', sha256(SOURCE_ZIP))


In [ ]:
# 2. Extract and pin the primitive source chain
release_extract = WORK / '_release_extract'
if release_extract.exists():
    shutil.rmtree(release_extract)
release_extract.mkdir()

with zipfile.ZipFile(SOURCE_ZIP) as zf:
    zf.extractall(release_extract)

nested = list(release_extract.rglob('SU6_DETERMINANT_ARCHIVAL_V2_BUNDLE.zip'))
if nested:
    with zipfile.ZipFile(nested[0]) as zf:
        zf.extractall(INPUT)
else:
    # Direct archival bundle.
    for item in release_extract.iterdir():
        destination = INPUT / item.name
        if item.is_dir():
            shutil.copytree(item, destination)
        else:
            shutil.copy2(item, destination)

matches = list(INPUT.rglob('SU6_DETERMINANT_ARCHIVAL_V2'))
if matches:
    BUNDLE = matches[0]
elif (INPUT / 'stage0.py').exists():
    BUNDLE = INPUT
else:
    raise FileNotFoundError('SU6_DETERMINANT_ARCHIVAL_V2 source directory was not found after extraction.')

required = [
    'stage0.py',
    'y4_connected_supports.json.gz',
    'y4_stage0_summary.json',
    'y4_sun_walled_brauer_fixed_rank.py',
    'y4_sun_stable_rank_stage1.py',
    'y4_sun_stable_ordered_words.json.gz',
    'y4_extracted_sources.zip',
]
for name in required:
    if not (BUNDLE / name).exists():
        raise FileNotFoundError(BUNDLE / name)

support_hash = sha256(BUNDLE / 'y4_connected_supports.json.gz')
assert support_hash == '5b3621cba76d4ac34b6ee89302399f07415db144735b07bbc926d0b370c4acdf'
summary = json.loads((BUNDLE / 'y4_stage0_summary.json').read_text())
assert summary['counts']['connected_support_multisets'] == 182440
assert summary['counts']['candidate_support_output_pairs'] == 895524

# The legacy fixed-rank source searches /content before cwd. Pin exact copies there in Colab.
if Path('/content').exists():
    for name in ('y4_sun_stable_rank_stage1.py', 'y4_sun_stable_ordered_words.json.gz', 'y4_extracted_sources.zip'):
        shutil.copy2(BUNDLE / name, Path('/content') / name)

provenance = {
    'source_zip': str(SOURCE_ZIP),
    'source_zip_sha256': sha256(SOURCE_ZIP),
    'bundle_dir': str(BUNDLE),
    'primitive_support_sha256': support_hash,
}
(OUTPUT / 'SOURCE_PROVENANCE.json').write_text(json.dumps(provenance, indent=2, sort_keys=True) + '\n')
print('PASS primitive Stage-0 source chain')
print('PASS 182,440 connected supports')
print('PASS 895,524 support/output candidates')
print('BUNDLE', BUNDLE)


In [ ]:
# 3. Install the exact process-isolated stage sources
stage_sources = {}
stage_sources['su4_explicit_core.py'] = "from __future__ import annotations\nimport json,gzip,itertools,math,importlib.util,zipfile,tempfile,time\nfrom pathlib import Path\nfrom collections import defaultdict,Counter\nfrom fractions import Fraction\nimport sympy as sp\nN=4; CUTS=(1,2,3); E=((1,0,0),(0,1,0),(0,0,1))\nB=Path('/mnt/data/su4_nb_build/SU6_DETERMINANT_ARCHIVAL_V2')\nspec=importlib.util.spec_from_file_location('st',B/'stage0.py');st=importlib.util.module_from_spec(spec);spec.loader.exec_module(st)\n# stage3j\nwith zipfile.ZipFile(B/'y4_extracted_sources.zip') as z:\n td=tempfile.TemporaryDirectory(); z.extractall(td.name); p=next(Path(td.name).rglob('stage3j.py'))\n spec=importlib.util.spec_from_file_location('j',p);j=importlib.util.module_from_spec(spec);spec.loader.exec_module(j)\nscan=json.load(open('/mnt/data/su4_nb_build/su4_scan_summary.json'))\n\ndef vadd(u,v):return tuple(u[i]+v[i] for i in range(3))\ndef pb(p):\n x=p[:3];a,b=p[3:];v1=vadd(x,E[a]);v3=vadd(x,E[b])\n return (((*x,a),+1,0,1),((*v1,b),+1,1,2),((*v3,a),-1,2,3),((*x,b),-1,3,0))\n\ndef eps4(vals):\n if len(set(vals))<4:return 0\n inv=sum(vals[i]>vals[j] for i in range(4) for jj in [range(i+1,4)] for j in jj)\n return -1 if inv%2 else 1\n\ndef tadd(*terms):\n d={}\n for coeff,a in terms:\n  coeff=sp.Rational(coeff)\n  for k,v in a.items():d[k]=d.get(k,0)+coeff*v\n return {k:sp.cancel(v) for k,v in d.items() if v}\ndef tscale(a,c):return {k:sp.cancel(c*v) for k,v in a.items() if c*v}\ndef tdot(a,b):return sum(sp.Rational(v)*b.get(k,0) for k,v in a.items())\ndef tswap(a,i,j):\n d={}\n for k,v in a.items():\n  q=list(k);q[i],q[j]=q[j],q[i];q=tuple(q);d[q]=d.get(q,0)+v\n return d\ndef tK(a,i,j,m):\n grouped={}\n for k,v in a.items():\n  if k[i]==k[j]:\n   other=tuple(k[x] for x in range(m) if x not in (i,j));grouped[other]=grouped.get(other,0)+v\n d={}\n for other,v in grouped.items():\n  for c in range(N):\n   q=[];it=iter(other)\n   for x in range(m):q.append(c if x in (i,j) else next(it))\n   d[tuple(q)]=v\n return d\n\ndef raw_basis(sig):\n active=[(e,t) for e,t in enumerate(sig) if t];types=[t for e,t in active]\n plus=[i for i,t in enumerate(types) if t==1];minus=[i for i,t in enumerate(types) if t==-1]\n r,s=len(plus),len(minus);out=[];names=[];m=r+s\n if r==s:\n  for perm in itertools.permutations(range(r)):\n   d={}\n   for pc in itertools.product(range(N),repeat=r):\n    key=[None]*m\n    for a,pos in enumerate(plus):key[pos]=pc[a]\n    for a,pos in enumerate(plus):key[minus[perm[a]]]=pc[a]\n    d[tuple(key)]=1\n   out.append(d);names.append('pair_'+''.join(map(str,perm)))\n elif (r,s)==(4,0) or (r,s)==(0,4):\n  d={k:eps4(k) for k in itertools.permutations(range(4))};out=[d];names=['epsilon']\n elif (r,s)==(5,1):\n  ma=minus[0]\n  for a,pick in enumerate(plus):\n   rem=[p for p in plus if p!=pick];d={}\n   for c in range(N):\n    for vals in itertools.permutations(range(4)):\n     key=[None]*m;key[pick]=c;key[ma]=c\n     for pos,v in zip(rem,vals):key[pos]=v\n     d[tuple(key)]=eps4(vals)\n   out.append(d);names.append(f'delta_plus_{a}_epsilon')\n elif (r,s)==(1,5):\n  pf=plus[0]\n  for a,pick in enumerate(minus):\n   rem=[q for q in minus if q!=pick];d={}\n   for c in range(N):\n    for vals in itertools.permutations(range(4)):\n     key=[None]*m;key[pf]=c;key[pick]=c\n     for pos,v in zip(rem,vals):key[pos]=v\n     d[tuple(key)]=eps4(vals)\n   out.append(d);names.append(f'delta_minus_{a}_epsilon')\n else:raise ValueError((sig,r,s))\n return active,out,names\n\ndef casimir_apply(a,types,pref):\n CF=sp.Rational(15,8);out=tscale(a,sum(pref)*CF);inds=[i for i,x in enumerate(pref) if x]\n for ii,i in enumerate(inds):\n  for k in inds[ii+1:]:\n   if types[i]==types[k]:out=tadd((1,out),(1,tswap(a,i,k)),(-sp.Rational(1,N),a))\n   else:out=tadd((1,out),(-1,tK(a,i,k,len(types))),(sp.Rational(1,N),a))\n return out\n\nclass ExplicitLibrary:\n def __init__(self):self.cache={}\n def get(self,sig):\n  sig=tuple(sig)\n  if sig in self.cache:return self.cache[sig]\n  active,raw,names=raw_basis(sig)\n  Graw=sp.Matrix([[tdot(a,b) for b in raw] for a in raw]);piv=Graw.rref()[1]\n  basis=[raw[i] for i in piv];bn=[names[i] for i in piv]\n  G=sp.Matrix([[tdot(a,b) for b in basis] for a in basis]);assert G.det()!=0\n  types=[t for e,t in active];mats=[]\n  for cut in CUTS:\n   acts=[casimir_apply(v,types,[e<=cut for e,t in active]) for v in basis]\n   H=sp.Matrix([[tdot(basis[i],acts[jj]) for jj in range(len(basis))] for i in range(len(basis))])\n   M=sp.simplify(G.inv()*H);assert sp.simplify(M.T*G-G*M)==sp.zeros(M.rows)\n   mats.append(M)\n  evs=[list(M.eigenvals()) for M in mats];paths=[]\n  for vals in itertools.product(*evs):\n   stack=sp.Matrix.vstack(*[M-v*sp.eye(M.rows) for M,v in zip(mats,vals)])\n   for vec in stack.nullspace():\n    # normalize coordinate first nonzero to 1\n    for x in vec:\n     if x:vec=sp.simplify(vec/x);break\n    ten={}\n    for c,b in zip(vec,basis):\n     if c:ten=tadd((1,ten),(c,b))\n    norm=sp.cancel(tdot(ten,ten));assert norm>0\n    paths.append({'history':tuple(sp.Rational(v) for v in vals),'tensor':ten,'norm':sp.Rational(norm),'basis_coeff':[str(x) for x in vec]})\n  assert len(paths)==G.rows,(sig,len(paths),G.rows)\n  # orthogonal completeness in ambient invariant space\n  for i,a in enumerate(paths):\n   for jj,b in enumerate(paths):\n    z=sp.cancel(tdot(a['tensor'],b['tensor']))\n    assert (z==0)==(i!=jj),(sig,i,jj,z)\n  rec={'active':active,'basis_names':bn,'gram':G,'paths':paths,'raw_rank':Graw.rank(),'raw_dim':Graw.rows}\n  self.cache[sig]=rec;return rec\n\n# Factor algebra, sparse exact color tables\ndef make_factor(scope,tensor):\n return (tuple(scope),{tuple(k):sp.Rational(v) for k,v in tensor.items() if v})\ndef multiply_factor(A,B):\n sa,ta=A;sb,tb=B; shared=[x for x in sa if x in set(sb)];su=tuple(sorted(set(sa)|set(sb)))\n pa={x:i for i,x in enumerate(sa)};pb={x:i for i,x in enumerate(sb)};pu={x:i for i,x in enumerate(su)}\n # index B by shared colors\n idx=defaultdict(list)\n for kb,vb in tb.items():idx[tuple(kb[pb[x]] for x in shared)].append((kb,vb))\n out=defaultdict(lambda:sp.Integer(0))\n for ka,va in ta.items():\n  sk=tuple(ka[pa[x]] for x in shared)\n  for kb,vb in idx.get(sk,[]):\n   ku=[None]*len(su)\n   for x in sa:ku[pu[x]]=ka[pa[x]]\n   for x in sb:ku[pu[x]]=kb[pb[x]]\n   out[tuple(ku)]+=va*vb\n return su,{k:sp.cancel(v) for k,v in out.items() if v}\ndef sum_out(A,var):\n s,t=A;i=s.index(var);ns=s[:i]+s[i+1:];out=defaultdict(lambda:sp.Integer(0))\n for k,v in t.items():out[k[:i]+k[i+1:]]+=v\n return ns,{k:sp.cancel(v) for k,v in out.items() if v}\ndef min_fill(scopes,nvar=24):\n adj=[set() for _ in range(nvar)]\n for s in scopes:\n  s=sorted(set(s))\n  for i,a in enumerate(s):\n   for b in s[i+1:]:adj[a].add(b);adj[b].add(a)\n alive={v for s in scopes for v in s};order=[];width=0\n while alive:\n  best=None\n  for v in alive:\n   ns=adj[v]&alive;l=list(ns);fill=sum(b not in adj[a] for i,a in enumerate(l) for b in l[i+1:]);cand=(fill,len(ns),v)\n   if best is None or cand<best[0]:best=(cand,v,ns)\n  _,v,ns=best;width=max(width,len(ns));l=list(ns)\n  for i,a in enumerate(l):\n   for b in l[i+1:]:adj[a].add(b);adj[b].add(a)\n  alive.remove(v);order.append(v)\n return tuple(order),width\n\ndef contract_choice(specs,choices,lib,order):\n factors=[];norm=sp.Integer(1);hist=[sp.Integer(0)]*3\n for (sig,rows,cols),pi in zip(specs,choices):\n  path=lib.get(sig)['paths'][pi];factors.append(make_factor(rows,path['tensor']));factors.append(make_factor(cols,path['tensor']));norm*=path['norm']\n  hist=[hist[c]+path['history'][c] for c in range(3)]\n for var in order:\n  sel=[f for f in factors if var in f[0]];factors=[f for f in factors if var not in f[0]]\n  if not sel:continue\n  cur=sel[0]\n  for f in sel[1:]:cur=multiply_factor(cur,f)\n  factors.append(sum_out(cur,var))\n cur=factors[0]\n for f in factors[1:]:cur=multiply_factor(cur,f)\n assert cur[0]==() and set(cur[1]).issubset({()})\n raw=cur[1].get((),0)/norm\n E0=sp.Rational(15,4); ds=[sp.cancel(E0-sp.Rational(1,2)*x) for x in hist]\n zero=[d for d in ds if d==0];non=[d for d in ds if d]\n if len(zero)==0:fold=1/(ds[0]*ds[1]*ds[2])\n elif len(zero)==1:\n  x,y=non;fold=-sp.Rational(1,2)*(1/(x*x*y)+1/(x*y*y))\n elif len(zero)==2:fold=sp.Rational(1,3)/(non[0]**3)\n else:fold=0\n return sp.cancel(raw),tuple(ds),sp.cancel(raw*fold)\n\n"
stage_sources['su4_stage_balanced.py'] = "\nimport sys, os, json, gzip, importlib.util, math\nfrom pathlib import Path\nfrom fractions import Fraction\n\nbundle_dir = Path(sys.argv[1]).resolve()\nout_path = Path(sys.argv[2]).resolve()\nfixed_path = bundle_dir / 'y4_sun_walled_brauer_fixed_rank.py'\nspec = importlib.util.spec_from_file_location('su4_balanced_fresh', fixed_path)\nfixed = importlib.util.module_from_spec(spec)\nassert spec.loader is not None\nos.chdir(bundle_dir)\nspec.loader.exec_module(fixed)\nassert Path(fixed.STABLE_SCRIPT).read_bytes() == (bundle_dir / 'y4_sun_stable_rank_stage1.py').read_bytes()\nassert Path(fixed.WORDS_PATH).read_bytes() == (bundle_dir / 'y4_sun_stable_ordered_words.json.gz').read_bytes()\nassert Path(fixed.SOURCES_ZIP).read_bytes() == (bundle_dir / 'y4_extracted_sources.zip').read_bytes()\n\namps, tops, word_orbits, words, qab = fixed.main(4, 0)\nrows=[]\nby_id={w['ordered_id']:w for w in words}\nfor ordered_id, word in by_id.items():\n    total=sum(Fraction(phase)*amps[key] for key,phase in word_orbits[ordered_id])\n    rows.append({\n        'ordered_id':ordered_id,\n        'ordered_insertions':word['ordered_insertions'],\n        'output':word['output'],\n        'canonical_complete_sum_odd':str(total),\n    })\nroot=fixed.j.build_root_kernel(rows)\nkernel=fixed.j.build_full_kernel(root)\nrecords=[{\n    'input_plane':list(key[0]),\n    'output_plane':list(key[1]),\n    'displacement':list(key[2]),\n    'weight':str(value),\n} for key,value in sorted(kernel.items())]\ncount_lib=fixed.LocalLibrary(4)\ncomputed_paths=sum(math.prod(len(count_lib.get(sig)) for sig,rows,cols in rec['specs']) for rec in tops.values())\nassert computed_paths==35130, computed_paths\npayload={\n    'qab':{k:(str(v) if isinstance(v,Fraction) else v) for k,v in qab.items()},\n    'counts':{'topologies':len(tops),'fusion_paths':computed_paths,'root_entries':len(root),'kernel_entries':len(kernel)},\n    'records':records,\n}\nout_path.write_text(json.dumps(payload,indent=2,sort_keys=True)+'\\n')\nprint('BALANCED SUBPROCESS OUTPUT',out_path)\n"
stage_sources['su4_stage_crosscheck.py'] = "from __future__ import annotations\nimport argparse,importlib.util,json,gzip,itertools,math,os\nfrom pathlib import Path\nfrom fractions import Fraction\nimport sympy as sp\nfrom su4_explicit_core import *\ndef load(path,name):spec=importlib.util.spec_from_file_location(name,path);m=importlib.util.module_from_spec(spec);assert spec.loader;spec.loader.exec_module(m);return m\ndef main():\n ap=argparse.ArgumentParser();ap.add_argument('bundle',type=Path);a=ap.parse_args();os.chdir(a.bundle);fixed=load(a.bundle/'y4_sun_walled_brauer_fixed_rank.py','fixed_check')\n words=json.load(gzip.open(a.bundle/'y4_sun_stable_ordered_words.json.gz','rt'))['words'];pl=fixed.LocalLibrary(4);tops,_=fixed.build_corpus(words,pl);el=ExplicitLibrary();items=[]\n for key,rec in tops.items():\n  pc=math.prod(len(pl.get(sig)) for sig,r,c in rec['specs'])\n  if pc in (1,2,4,6,8,12,16,24):items.append((pc,str(key),rec))\n items.sort();indices=sorted(set(round(i*(len(items)-1)/29) for i in range(30)));checked=0\n for idx in indices:\n  pc,_,rec=items[idx];pt=Fraction(0);et=sp.Integer(0)\n  for ch in itertools.product(*[range(len(pl.get(sig))) for sig,r,c in rec['specs']]):\n   raw,en=fixed.contract_choice(rec['specs'],ch,pl,rec['order']);pt+=raw*fixed.folded_coeff(en,4)\n  order,_=min_fill([s for sig,r,c in rec['specs'] for s in (r,c)])\n  for ch in itertools.product(*[range(len(el.get(sig)['paths'])) for sig,r,c in rec['specs']]):\n   raw,ds,val=contract_choice(rec['specs'],ch,el,order);et+=val\n  assert sp.cancel(et)==sp.Rational(pt.numerator,pt.denominator);checked+=1\n assert checked==30;print('PASS 30 BALANCED TOPOLOGY CROSS-CHECKS')\nif __name__=='__main__':main()\n"
stage_sources['su4_stage_exception.py'] = "from __future__ import annotations\nimport argparse,json,importlib.util,zipfile,tempfile,time,itertools,math,hashlib\nfrom pathlib import Path\nfrom collections import defaultdict,Counter\nfrom fractions import Fraction\nimport sympy as sp\nfrom su4_explicit_core import *\n\ndef load(path,name):\n spec=importlib.util.spec_from_file_location(name,path);m=importlib.util.module_from_spec(spec);assert spec.loader;spec.loader.exec_module(m);return m\n\ndef main():\n ap=argparse.ArgumentParser();ap.add_argument('bundle',type=Path);ap.add_argument('scan',type=Path);ap.add_argument('output',type=Path);a=ap.parse_args();a.output.mkdir(parents=True,exist_ok=True)\n global st,j,scan\n st=load(a.bundle/'stage0.py','su4_stage0_exc')\n td=tempfile.TemporaryDirectory(prefix='su4_stage3j_')\n with zipfile.ZipFile(a.bundle/'y4_extracted_sources.zip') as z:z.extractall(td.name)\n j=load(next(Path(td.name).rglob('stage3j.py')),'su4_stage3j_exc')\n scan=json.loads(a.scan.read_text())\n # corpus\n lib=ExplicitLibrary();tops={};word_orbits=defaultdict(list)\n for rr in scan['records']:\n  word=tuple(tuple(x) for x in rr['ordered_insertions']);out=tuple(rr['output']);signs=tuple(rr['signs']);factors=[st.ROOT,*word,out];eff=signs[:5]+(-signs[5],);links=defaultdict(list)\n  for ei,p in enumerate(factors):\n   for edge,(link,inc,sc,ec) in enumerate(pb(p)):\n    token=eff[ei]*inc;rv=4*ei+(sc if inc==1 else ec);cv=4*ei+(ec if inc==1 else sc)\n    links[link].append((ei,edge,token,rv,cv))\n  groups=[];specs=[]\n  for link,occ in sorted(links.items()):\n   occ=tuple(sorted(occ));groups.append(occ);sig=[0]*6\n   for ei,edge,t,rv,cv in occ:sig[ei]=t\n   sig=tuple(sig);lib.get(sig);specs.append((sig,tuple(x[3] for x in occ),tuple(x[4] for x in occ)))\n  key=(signs[0]*signs[5],tuple(sorted(groups)))\n  if key not in tops:\n   order,width=min_fill([x for s,r,c in specs for x in (r,c)]);tops[key]={'specs':specs,'order':order,'width':width}\n  word_orbits[(word,out)].append((key,signs[0]*signs[5]))\n print('library',len(lib.cache),'tops',len(tops),'words',len(word_orbits),flush=True)\n # contract\n amps={};total_paths=0;t0=time.time();denhist=Counter()\n for ti,(key,rec) in enumerate(sorted(tops.items(),key=lambda kv:math.prod(len(lib.get(s)['paths']) for s,r,c in kv[1]['specs'])),1):\n  ranges=[range(len(lib.get(s)['paths'])) for s,r,c in rec['specs']];tot=sp.Integer(0);pc=0\n  for choices in itertools.product(*ranges):\n   raw,ds,val=contract_choice(rec['specs'],choices,lib,rec['order']);tot+=val;pc+=1;denhist[tuple(str(x) for x in ds)]+=1\n  amps[key]=sp.cancel(tot);total_paths+=pc\n  if ti%10==0 or ti==len(tops):print('top',ti,'paths',total_paths,'sec',time.time()-t0,flush=True)\n # rows for stage3j\n rows=[]\n for oid,((word,out),orbs) in enumerate(sorted(word_orbits.items()),1):\n  total=sum(sp.Rational(ph)*amps[key] for key,ph in orbs)\n  rows.append({'ordered_id':f'su4_exc_{oid:04d}','ordered_insertions':[list(x) for x in word],'output':list(out),'canonical_complete_sum_odd':str(sp.cancel(total))})\n root=j.build_root_kernel(rows);full=j.build_full_kernel(root)\n G=j.symbol_at_parity(full,(1,1,1));assert all(G[a][b]==(G[0][0] if a==b else 0) for a in range(3) for b in range(3)),G\n q=G[0][0]\n def corr(ph):return j.rayleigh(j.flat_vector_at_parity(ph),j.symbol_at_parity(full,ph))\n xvals={corr((-1,1,1)),corr((1,-1,1)),corr((1,1,-1))};mvals={corr((-1,-1,1)),corr((-1,1,-1)),corr((1,-1,-1))};assert len(xvals)==len(mvals)==1\n x=next(iter(xvals));m=next(iter(mvals));r=corr((-1,-1,-1));A=x-q;Bv=2*(m-x);assert Bv==r-x\n print('EXCEPTION', {k:str(v) for k,v in {'q':q,'X':x,'M':m,'R':r,'A':A,'B':Bv,'bandwidth':A+Bv}.items()})\n print('root',len(root),'full',len(full),'paths',total_paths,'denhist top',denhist.most_common(10))\n # exact projected Laurent residual\n from fractions import Fraction as F\n ZERO=(0,0,0)\n def clean(p):return {e:c for e,c in p.items() if c}\n def add(*ps):\n  d=defaultdict(F)\n  for p in ps:\n   for e,c in p.items():d[e]+=c\n  return clean(d)\n def mul(a,b):\n  d=defaultdict(F)\n  for e,c in a.items():\n   for f,v in b.items():d[tuple(e[i]+f[i] for i in range(3))]+=c*v\n  return clean(d)\n def adj(a):return {tuple(-x for x in e):v for e,v in a.items()}\n def shift(a,r,c=F(1)):return {tuple(e[i]+r[i] for i in range(3)):c*v for e,v in a.items()}\n def de(i,n=1):q=[0,0,0];q[i]=n;return tuple(q)\n def mono(e,c=F(1)):return {e:c}\n psi=(add(mono(de(2)),mono(ZERO,F(-1))),add(mono(ZERO),mono(de(1),F(-1))),add(mono(de(0)),mono(ZERO,F(-1))))\n planes=((0,1),(0,2),(1,2));pi={p:i for i,p in enumerate(planes)};la=defaultdict(F)\n for (ip,op,disp),w in full.items():\n  aa=pi[ip];bb=pi[op]\n  for e,c in shift(mul(adj(psi[bb]),psi[aa]),disp,w).items():la[e]+=c\n for aa in range(3):\n  for e,c in mul(adj(psi[aa]),psi[aa]).items():la[e]-=q*c\n assert clean(la)=={}\n def krecs(k):return [{'input_plane':list(x[0]),'output_plane':list(x[1]),'displacement':list(x[2]),'weight':str(v)} for x,v in sorted(k.items())]\n local=[]\n for sig,rec in sorted(lib.cache.items()):\n  local.append({'signature':list(sig),'r_s':[sig.count(1),sig.count(-1)],'raw_dimension':rec['raw_dim'],'physical_rank':rec['raw_rank'],'gram_determinant':str(rec['gram'].det()),'histories':[{'casimirs':[str(x) for x in p['history']],'norm':str(p['norm'])} for p in rec['paths']]})\n lc={'meta':{'version':'2026-06-14-su4-local-epsilon-library-v1'},'counts':{'local_signatures':len(lib.cache),'exceptional_trace_topologies':len(tops),'exceptional_fusion_paths':total_paths},'signatures':local,'gates':{'passed':True}}\n (a.output/'SU4_LOCAL_EPSILON_LIBRARY_CERTIFICATE.json').write_text(json.dumps(lc,indent=2,sort_keys=True)+'\\n')\n result={'meta':{'version':'2026-06-14-su4-exceptional-contraction-v1'},'counts':{'ordered_transitions':len(word_orbits),'trace_topologies':len(tops),'fusion_paths':total_paths,'root_entries':len(root),'kernel_entries':len(full)},'results':{'delta_q4':str(q),'delta_A4':str(A),'delta_B4':str(Bv)},'rows':rows,'kernel':krecs(full)}\n assert result['counts']=={'ordered_transitions':76,'trace_topologies':96,'fusion_paths':1806,'root_entries':13,'kernel_entries':39}\n assert q==F(-304746539168,160249753125) and A==0 and Bv==0\n (a.output/'SU4_EXCEPTIONAL_CONTRACTION_CERTIFICATE.json').write_text(json.dumps(result,indent=2,sort_keys=True)+'\\n')\n (a.output/'y4_su4_exceptional_kernel_39.json').write_text(json.dumps({'meta':result['meta'],'records':krecs(full)},indent=2,sort_keys=True)+'\\n')\n print('ALL SU4 EXCEPTIONAL CONTRACTION GATES PASS')\n\nif __name__=='__main__':main()\n"
stage_sources['su4_stage_finalize.py'] = "from __future__ import annotations\nimport argparse,json,hashlib,zipfile,tempfile,importlib.util\nfrom pathlib import Path\nfrom fractions import Fraction as F\nfrom collections import defaultdict\ndef load(path,name):spec=importlib.util.spec_from_file_location(name,path);m=importlib.util.module_from_spec(spec);assert spec.loader;spec.loader.exec_module(m);return m\ndef main():\n ap=argparse.ArgumentParser();ap.add_argument('bundle',type=Path);ap.add_argument('balanced',type=Path);ap.add_argument('exception',type=Path);ap.add_argument('output',type=Path);a=ap.parse_args();a.output.mkdir(parents=True,exist_ok=True)\n td=tempfile.TemporaryDirectory();zipfile.ZipFile(a.bundle/'y4_extracted_sources.zip').extractall(td.name);j=load(next(Path(td.name).rglob('stage3j.py')),'jfinal')\n bp=json.loads(a.balanced.read_text());ep=json.loads(a.exception.read_text());bal={(tuple(r['input_plane']),tuple(r['output_plane']),tuple(r['displacement'])):F(r['weight']) for r in bp['records']};exc={(tuple(r['input_plane']),tuple(r['output_plane']),tuple(r['displacement'])):F(r['weight']) for r in ep['kernel']};full={k:bal.get(k,F(0))+exc.get(k,F(0)) for k in set(bal)|set(exc)};full={k:v for k,v in full.items() if v};assert len(full)==189\n for (ip,op,d),v in full.items():assert full.get((op,ip,tuple(-x for x in d)),F(0))==v\n def corr(ph):return j.rayleigh(j.flat_vector_at_parity(ph),j.symbol_at_parity(full,ph))\n G=j.symbol_at_parity(full,(1,1,1));q=G[0][0];X=corr((-1,1,1));M=corr((-1,-1,1));R=corr((-1,-1,-1));A=X-q;B=2*(M-X);assert B==R-X\n assert q==F(-162485785670299274695454289332603,121294607143027203361265133093750);assert A==F(32,675);assert B==F(3601925923737103752887,70481696720359496343750)\n ZERO=(0,0,0)\n def clean(p):return {e:c for e,c in p.items() if c}\n def add(*ps):\n  d=defaultdict(F)\n  for p in ps:\n   for e,c in p.items():d[e]+=c\n  return clean(d)\n def scale(p,s):return clean({e:s*c for e,c in p.items()})\n def mul(a,b):\n  d=defaultdict(F)\n  for e,c in a.items():\n   for f,v in b.items():d[tuple(e[i]+f[i] for i in range(3))]+=c*v\n  return clean(d)\n def adj(p):return {tuple(-x for x in e):c for e,c in p.items()}\n def shift(p,d,s=F(1)):return {tuple(e[i]+d[i] for i in range(3)):s*c for e,c in p.items()}\n def de(i,n=1):e=[0,0,0];e[i]=n;return tuple(e)\n def mono(e,c=F(1)):return {e:c}\n psi=(add(mono(de(2)),mono(ZERO,F(-1))),add(mono(ZERO),mono(de(1),F(-1))),add(mono(de(0)),mono(ZERO,F(-1))))\n def contracted(kern,qq):\n  planes=((0,1),(0,2),(1,2));pi={p:i for i,p in enumerate(planes)};out=defaultdict(F)\n  for (ip,op,d),w in kern.items():\n   aa=pi[ip];bb=pi[op]\n   for e,c in shift(mul(adj(psi[bb]),psi[aa]),d,w).items():out[e]+=c\n  for aa in range(3):\n   for e,c in mul(adj(psi[aa]),psi[aa]).items():out[e]-=qq*c\n  return clean(out)\n qexc=F(ep['results']['delta_q4']);assert contracted(exc,qexc)=={}\n def lap(i):return {ZERO:F(2),de(i):F(-1),de(i,-1):F(-1)}\n ls=[lap(i) for i in range(3)];terms=[scale(mul(ls[i],ls[i]),A/F(4)) for i in range(3)]+[scale(mul(ls[i],ls[k]),B/F(4)) for i in range(3) for k in range(i+1,3)];assert contracted(full,q)==add(*terms);assert len(contracted(full,q))==25\n records=[{'input_plane':list(k[0]),'output_plane':list(k[1]),'displacement':list(k[2]),'weight':str(v)} for k,v in sorted(full.items())];sem=hashlib.sha256(json.dumps(records,sort_keys=True,separators=(',',':')).encode()).hexdigest()\n result={'meta':{'version':'2026-06-14-su4-complete-fourth-order-v1','rank':4,'record_count':189,'kernel_semantic_sha256':sem},'counts':{'connected_supports_scanned':182440,'candidate_support_output_pairs':895524,'balanced_global_fusion_paths':35130,'exceptional_global_fusion_paths':1806,'exceptional_ordered_transitions':76,'exceptional_trace_topologies':96},'results':{'q4_balanced':bp['qab']['q'],'q4_exceptional':str(qexc),'q4':str(q),'X4':str(X),'M4':str(M),'R4':str(R),'A4':str(A),'B4':str(B),'bandwidth':str(A+B)},'gates':{'exact_Hermiticity':True,'parity_identity':True,'exceptional_full_Laurent_residual_zero':True,'complete_25_point_SOS_identity':True,'passed':True}}\n (a.output/'y4_su4_corrected_full_kernel_189.json').write_text(json.dumps({'meta':result['meta'],'records':records,'semantic_sha256':sem},indent=2,sort_keys=True)+'\\n');(a.output/'SU4_COMPLETE_FOURTH_ORDER_CERTIFICATE.json').write_text(json.dumps(result,indent=2,sort_keys=True)+'\\n')\n theorem=f'''# SU(4) fourth-order determinant-sector completion\\n\\n**Status:** PASS\\n\\nThe complete modulo-four scan and exact epsilon contraction give\\n\\n$$\\\\Delta q_4={qexc},\\\\qquad \\\\Delta A_4=\\\\Delta B_4=0.$$\\n\\nThe final coefficients are\\n\\n$$q_4={q},\\\\qquad A_4={A},\\\\qquad B_4={B},$$\\n\\nwith exact bandwidth $A_4+B_4={A+B}>0$. The full 25-point Laurent identity proves a unique minimum at $\\\\Gamma$ and unique maximum at $R$.\\n\\nKernel semantic SHA-256: `{sem}`.\\n''';(a.output/'SU4_COMPLETE_FOURTH_ORDER_THEOREM.md').write_text(theorem)\n print('ALL SU4 COMPLETE FOURTH-ORDER GATES PASS');print(json.dumps(result['results'],indent=2));print('HASH',sem)\nif __name__=='__main__':main()\n"
stage_sources['su4_stage_scan.py'] = "from __future__ import annotations\nimport argparse,gzip,json,itertools,importlib.util,time,hashlib\nfrom pathlib import Path\nfrom collections import Counter\nSIGNS=tuple(itertools.product((-1,1),repeat=6));FULL=(1<<64)-1\n\ndef sha256(p):\n h=hashlib.sha256();h.update(p.read_bytes());return h.hexdigest()\ndef load(path,name):\n spec=importlib.util.spec_from_file_location(name,path);m=importlib.util.module_from_spec(spec);assert spec.loader;spec.loader.exec_module(m);return m\n\ndef main():\n ap=argparse.ArgumentParser();ap.add_argument('bundle',type=Path);ap.add_argument('output',type=Path);a=ap.parse_args();a.output.mkdir(parents=True,exist_ok=True)\n st=load(a.bundle/'stage0.py','su4_stage0_scan')\n supports=json.load(gzip.open(a.bundle/'y4_connected_supports.json.gz','rt'))['supports'];supports=[tuple(tuple(int(x) for x in p) for p in ms) for ms in supports]\n assert sha256(a.bundle/'y4_connected_supports.json.gz')=='5b3621cba76d4ac34b6ee89302399f07415db144735b07bbc926d0b370c4acdf'\n def rows_for(word,out):\n  rows={};factors=(st.ROOT,)+tuple(word)+(out,)\n  for col,p in enumerate(factors):\n   ext=-1 if col==5 else 1\n   for link,inc in st.boundary(p):rows.setdefault(link,[0]*6)[col]+=ext*inc\n  return tuple(sorted((tuple(k),tuple(v)) for k,v in rows.items()))\n cache={}\n def masks(rows):\n  exact=FULL;mod=FULL\n  for _,row in rows:\n   if row not in cache:\n    em=mm=0\n    for i,s in enumerate(SIGNS):\n     x=sum(row[j]*s[j] for j in range(6))\n     if x==0:em|=1<<i\n     if x%4==0:mm|=1<<i\n    cache[row]=(em,mm)\n   em,mm=cache[row];exact&=em;mod&=mm\n   if not mod:break\n  return exact,mod\n def reps(mask):\n  vals=[SIGNS[i] for i in range(64) if mask>>i&1]\n  return sorted({min(s,tuple(-x for x in s)) for s in vals})\n classes={};cand=raw=0;t0=time.time()\n for idx,ms in enumerate(supports,1):\n  for out in st.candidate_outputs(ms):\n   cand+=1;ex,mo=masks(rows_for(ms,out));extra=mo&~ex\n   if extra:raw+=1;classes[st.canonical_support_output(ms,out)]=1\n  if idx%40000==0:print('supports',idx,'classes',len(classes),flush=True)\n keys=set()\n for ms,out in classes:\n  for word in set(itertools.permutations(ms)):keys.add(st.canonical_ordered_transition(word,out))\n records=[];fh=Counter();ph=Counter()\n for word,out in sorted(keys):\n  rows=rows_for(word,out);ex,mo=masks(rows);extra=mo&~ex\n  for signs in reps(extra):\n   full=[];prefix=[]\n   for link,row in rows:\n    toks=tuple(row[j]*signs[j] for j in range(6));act=[x for x in toks if x];r=sum(x>0 for x in act);s=sum(x<0 for x in act)\n    if r!=s:fh[(r,s)]+=1;full.append({'link':list(link),'signature':[r,s],'tokens':list(toks)})\n    cuts=[]\n    for cut in (1,2,3):\n     pr=[toks[j] for j in range(cut+1) if toks[j]];rr=sum(x>0 for x in pr);ss=sum(x<0 for x in pr)\n     if abs(rr-ss)>=4:ph[(cut,rr,ss)]+=1;cuts.append([cut,rr,ss])\n    if cuts:prefix.append({'link':list(link),'cuts':cuts})\n   records.append({'ordered_insertions':[list(x) for x in word],'output':list(out),'signs':list(signs),'phase':signs[0]*signs[5],'full_exceptional':full,'prefix_exceptional':prefix})\n counts={'connected_supports':len(supports),'candidate_support_output_pairs':cand,'raw_exceptional_support_output_pairs':raw,'canonical_exceptional_support_output_classes':len(classes),'exceptional_ordered_transitions':len({(tuple(tuple(x) for x in r['ordered_insertions']),tuple(r['output'])) for r in records}),'ordered_word_assignment_orbits':len(records),'records_with_prefix_determinant_sector':sum(bool(r['prefix_exceptional']) for r in records)}\n expected={'connected_supports':182440,'candidate_support_output_pairs':895524,'raw_exceptional_support_output_pairs':21,'canonical_exceptional_support_output_classes':21,'exceptional_ordered_transitions':76,'ordered_word_assignment_orbits':156,'records_with_prefix_determinant_sector':28};assert counts==expected\n assert fh==Counter({(0,4):270,(4,0):270,(1,5):42,(5,1):42});assert ph==Counter({(3,0,4):32,(3,4,0):32})\n payload={'meta':{'version':'2026-06-14-su4-mod4-prefix-scan-v1','supports_sha256':sha256(a.bundle/'y4_connected_supports.json.gz')},'counts':counts,'full_signature_histogram':{str(k):v for k,v in sorted(fh.items())},'prefix_signature_histogram':{str(k):v for k,v in sorted(ph.items())},'records':records,'elapsed_seconds':time.time()-t0}\n out=a.output/'SU4_EXCEPTIONAL_MOD4_SCAN_CERTIFICATE.json';out.write_text(json.dumps(payload,indent=2,sort_keys=True)+'\\n');print('ALL SU4 SCAN GATES PASS',out)\nif __name__=='__main__':main()\n"
for name, source in stage_sources.items():
    (STAGES / name).write_text(source)
print('WROTE', len(stage_sources), 'stage sources to', STAGES)

In [ ]:
# 4. Subprocess runner: streams output and freezes a log for every phase
def run_stage(label: str, script: str, *arguments: object, needs_core: bool = False):
    command = [sys.executable, str(STAGES / script), *[str(x) for x in arguments]]
    environment = os.environ.copy()
    if needs_core:
        environment['PYTHONPATH'] = str(STAGES) + os.pathsep + environment.get('PYTHONPATH', '')
    log_path = LOGS / f'{label}.log'
    print('\n' + '=' * 100)
    print(label)
    print('COMMAND:', ' '.join(command))
    print('=' * 100)
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(
            command,
            cwd=str(WORK),
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{label} failed with return code {return_code}; see {log_path}')
    print('PASS', label, 'LOG', log_path)


## Stage A — balanced fixed-rank contraction

This fresh process runs the complete 35,130-path balanced walled-Brauer engine at $N=4$ and serializes its 189-record kernel.

In [ ]:
run_stage(
    '01_balanced_N4',
    'su4_stage_balanced.py',
    BUNDLE,
    OUTPUT / 'SU4_BALANCED_N4_CERTIFICATE.json',
)


## Stage B — primitive modulo-four scan

This scans the complete primitive corpus and records every full-link and intermediate-cut determinant sector.

In [ ]:
run_stage(
    '02_mod4_scan',
    'su4_stage_scan.py',
    BUNDLE,
    OUTPUT,
)


## Stage C — exact epsilon-sector contraction

The local $(4,0)$ and $(0,4)$ epsilon tensors and the rank-four quotients of the raw $(5,1)$ and $(1,5)$ bases are constructed exactly. All exceptional intermediate Casimir histories and 1,806 fusion paths are then contracted.

In [ ]:
run_stage(
    '03_exceptional_epsilon',
    'su4_stage_exception.py',
    BUNDLE,
    OUTPUT / 'SU4_EXCEPTIONAL_MOD4_SCAN_CERTIFICATE.json',
    OUTPUT,
    needs_core=True,
)


## Stage D — independent convention regression

Thirty balanced trace topologies are evaluated by both the original partition-loop engine and the explicit color/Fierz engine. Exact equality checks the tensor conventions used in the exceptional sector.

In [ ]:
run_stage(
    '04_cross_engine_regression',
    'su4_stage_crosscheck.py',
    BUNDLE,
    needs_core=True,
)


## Stage E — corrected kernel and SOS theorem

This combines the disjoint balanced and exceptional sectors, checks exact Hermiticity, constructs the corrected 189-record kernel, and proves the full 25-point Laurent/SOS identity.

In [ ]:
run_stage(
    '05_finalize',
    'su4_stage_finalize.py',
    BUNDLE,
    OUTPUT / 'SU4_BALANCED_N4_CERTIFICATE.json',
    OUTPUT / 'SU4_EXCEPTIONAL_CONTRACTION_CERTIFICATE.json',
    OUTPUT,
)


In [ ]:
# 10. Read and display the final exact result
CERTIFICATE = json.loads((OUTPUT / 'SU4_COMPLETE_FOURTH_ORDER_CERTIFICATE.json').read_text())
RESULT = CERTIFICATE['results']
print(json.dumps(RESULT, indent=2))
print('\nKernel semantic SHA-256:', CERTIFICATE['meta']['kernel_semantic_sha256'])
assert CERTIFICATE['meta']['kernel_semantic_sha256'] == 'e8bc5badc026d9874af56cd9ded47c4f0b0833597cb13ae5fd5e618a113dfeb9'
assert CERTIFICATE['gates']['passed']


## Certified result

The exceptional determinant/trace-identity sector produces a momentum-rigid projected correction,

\[
\Delta q_4=-\frac{304746539168}{160249753125},\qquad
\Delta A_4=\Delta B_4=0.
\]

The complete fourth-order coefficients are

\[
q_4=-\frac{162485785670299274695454289332603}
{121294607143027203361265133093750},
\]

\[
A_4=\frac{32}{675},\qquad
B_4=\frac{3601925923737103752887}
{70481696720359496343750},
\]

with positive bandwidth

\[
\Delta c_{4,4}=A_4+B_4
=\frac{2314426811641505637629}
{23493898906786498781250}>0.
\]

The exact SOS identity proves a unique fourth-order minimum at $\Gamma$ and a unique maximum at $R$.

In [ ]:
# 11. Freeze sources, logs, hashes, and all certificates into one archival ZIP
source_out = OUTPUT / 'sources'
log_out = OUTPUT / 'logs'
source_out.mkdir(exist_ok=True)
log_out.mkdir(exist_ok=True)
for path in sorted(STAGES.glob('*.py')):
    shutil.copy2(path, source_out / path.name)
for path in sorted(LOGS.glob('*.log')):
    shutil.copy2(path, log_out / path.name)

readme = (
    '# SU(4) exact fourth-order completion bundle\n\n'
    'Run order:\n'
    '1. su4_stage_balanced.py\n'
    '2. su4_stage_scan.py\n'
    '3. su4_stage_exception.py\n'
    '4. su4_stage_crosscheck.py\n'
    '5. su4_stage_finalize.py\n\n'
    'Final corrected kernel semantic SHA-256:\n'
    f"{CERTIFICATE['meta']['kernel_semantic_sha256']}\n\n"
    'All exact gates passed in the generating notebook.\n'
)
(OUTPUT / 'README.md').write_text(readme)

manifest_lines = []
for path in sorted(p for p in OUTPUT.rglob('*') if p.is_file() and p.name != 'SHA256SUMS.txt'):
    manifest_lines.append(f'{sha256(path)}  {path.relative_to(OUTPUT)}')
(OUTPUT / 'SHA256SUMS.txt').write_text('\n'.join(manifest_lines) + '\n')

ARCHIVE = BASE / 'SU4_COMPLETE_FOURTH_ORDER_BUNDLE.zip'
if ARCHIVE.exists():
    ARCHIVE.unlink()
with zipfile.ZipFile(ARCHIVE, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(p for p in OUTPUT.rglob('*') if p.is_file()):
        zf.write(path, arcname=f'SU4_COMPLETE_FOURTH_ORDER/{path.relative_to(OUTPUT)}')

with zipfile.ZipFile(ARCHIVE) as zf:
    bad = zf.testzip()
assert bad is None
print('BUNDLE', ARCHIVE)
print('BUNDLE SHA-256', sha256(ARCHIVE))
print('ZIP INTEGRITY PASS')

if IN_COLAB:
    from google.colab import files
    files.download(str(ARCHIVE))
